# Repaso 3: Datos no estructurados
### Prueba de conocimientos y deducciones sobre texto e imágenes sintéticos

**Realizado por:** Joaquín Bermeo

## Propósito de la actividad

Consolidar lo estudiado sobre datos no estructurados trabajando con mensajes de texto de clientes y con imágenes representadas como matrices. La actividad repasa el preprocesamiento de texto, la tokenización, las frecuencias, la extracción de entidades con expresiones regulares, la representación numérica (bolsa de palabras, TF-IDF y similitud coseno), la conversión de texto en variables estructuradas y la extracción de características de imágenes, para terminar con una **deducción razonada** que incluya las limitaciones de los métodos.

## Contenidos que se repasan

- Flujo de trabajo con datos no estructurados: obtener, preprocesar, representar, analizar (notebook de datos no estructurados).
- Texto: normalización, frecuencias, TF-IDF, similitud coseno, expresiones regulares.
- Imágenes: matriz de píxeles, brillo, color, contraste, huella visual.
- Uso posterior de las variables derivadas como columnas de una tabla (notebooks de datos estructurados y de tipos de datos).

## Instrucciones

- Trabaje las secciones en orden: varios ejercicios reutilizan resultados o funciones de ejercicios anteriores.
- Registre cada resultado en la variable `respuesta_N` (donde N es el número del ejercicio), respetando exactamente el formato pedido.
- No modifique las celdas de preparación de datos; trabaje siempre sobre copias de los datos originales.
- Los ejercicios piden **cálculos y deducciones**, no gráficos.
- Los redondeos se indican en cada enunciado. Ante un empate se aplica la regla que se indique; si no se indica ninguna, no hay empates.
- Puntaje total de la prueba: **25 puntos**.

## Preparación de los datos

Los datos son **sintéticos**: `documentos` es una lista de 150 mensajes de clientes a un servicio de soporte de una tienda en línea, y `imagenes` es una lista de 5 imágenes de 64 x 64 píxeles (matrices de números). Se generan siempre con el mismo contenido.

In [1]:
import os, re, json, math, warnings
from collections import Counter
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [2]:
def generar_documentos(n=150, seed=7):
    rng = np.random.default_rng(seed)
    plantillas = {
        "logistica": ["El pedido llegó con retraso de varios días", "La entrega fue rápida y el paquete llegó en perfecto estado",
                      "El paquete llegó dañado por el envío", "Todavía no recibo mi encomienda y nadie responde"],
        "pagos": ["Me cobraron dos veces con la tarjeta", "El pago fue rechazado pero el dinero se descontó",
                  "El reembolso llegó rápido y sin problemas", "No recibí la factura del cobro"],
        "producto": ["El producto es excelente y cumple lo prometido", "El artículo llegó defectuoso y no enciende",
                     "La calidad es mala y el material es frágil", "Es un producto bueno y funciona perfecto"],
        "atencion": ["El asesor fue amable y resolvió mi consulta", "La atención fue pésima y demoraron demasiado",
                     "Nadie respondió mis mensajes en el chat de soporte", "Excelente atención del equipo de soporte"],
    }
    todas = [s for lista in plantillas.values() for s in lista]
    docs = []
    for _ in range(n):
        partes = [todas[int(i)] for i in rng.choice(len(todas), size=int(rng.integers(1, 4)), replace=False)]
        if rng.random() < .80:
            partes.append(f"Mi pedido es PED-{int(rng.integers(10000, 99999))}")
        if rng.random() < .35:
            partes.append(f"Mi correo es usuario{int(rng.integers(1, 60))}@correo.ec")
        if rng.random() < .30:
            partes.append("Mi teléfono es 09" + "".join(str(int(x)) for x in rng.integers(0, 10, 8)))
        if rng.random() < .30:
            partes.append(f"Compré el {int(rng.integers(1, 29)):02d}/{int(rng.integers(1, 13)):02d}/2025")
        if rng.random() < .10:
            partes.append(f"Detalle en https://soporte.tienda.ec/ticket/{int(rng.integers(100, 999))}")
        texto = ". ".join(partes) + "."
        docs.append(texto.upper() if rng.random() < .10 else texto)
    return docs


def generar_imagenes(k=5, seed=3, tam=64):
    rng = np.random.default_rng(seed)
    imgs = []
    for _ in range(k):
        c1, c2 = rng.integers(10, 246, 3), rng.integers(10, 246, 3)
        t = np.linspace(0, 1, tam)[None, :, None]
        base = (c1 * (1 - t) + c2 * t) * np.ones((tam, 1, 1))
        y0, x0 = (int(v) for v in rng.integers(5, 30, 2))
        h, w = (int(v) for v in rng.integers(10, 30, 2))
        base[y0:y0 + h, x0:x0 + w] = rng.integers(0, 256, 3)
        base += rng.normal(0, int(rng.integers(2, 15)), base.shape)
        imgs.append(np.clip(base, 0, 255).astype(np.uint8))
    return imgs


documentos = generar_documentos()
imagenes = generar_imagenes()
print("Documentos:", len(documentos), "| Imágenes:", len(imagenes), "de forma", imagenes[0].shape)
for i in range(3):
    print(f"[{i}] {documentos[i]}")

Documentos: 150 | Imágenes: 5 de forma (64, 64, 3)
[0] El producto es excelente y cumple lo prometido. Nadie respondió mis mensajes en el chat de soporte. La calidad es mala y el material es frágil. Mi pedido es PED-14997. Mi teléfono es 0934817148. Compré el 09/09/2025.
[1] El artículo llegó defectuoso y no enciende.
[2] El artículo llegó defectuoso y no enciende. El asesor fue amable y resolvió mi consulta. Mi pedido es PED-13954.


In [3]:
# Constantes que se utilizan en los ejercicios
# STOPWORDS: palabras vacías generales mas palabras de plantilla del servicio (ped, pedido, correo, telefono, compre, detalle)
STOPWORDS = set("""a al algo algunas algunos ante antes aquí así aún además allí bajo cada como cómo con contra cual cuales cuando de del desde donde durante e el él ella ellas ellos en entre era eran eres es esa esas ese eso esos esta está estaba estaban están estar este esto estos fue fueron ha había habían han hasta hay he hace hacia la las le les lo los más me mi mis mucho muy nada ni no nos nosotros o os otra otras otro otros para pero poco por porque puede pueden qué que quien quienes se sea sean según ser si sí sido siendo sin sobre solo son soy su sus también tan tanto te tiene tienen todo todos tras tu tus un una uno unos vez veces y ya yo ped pedido correo teléfono compré detalle""".split())
POSITIVAS = {"rápida", "rápido", "perfecto", "excelente", "amable", "resolvió", "cumple", "bueno", "funciona"}
NEGATIVAS = {"retraso", "dañado", "nadie", "rechazado", "descontó", "defectuoso", "mala", "frágil", "pésima", "demoraron"}
print("Stopwords:", len(STOPWORDS), "| positivas:", len(POSITIVAS), "| negativas:", len(NEGATIVAS))

Stopwords: 139 | positivas: 9 | negativas: 10


# Ejercicios

## Ejercicio 1. Exploración del corpus (1 punto)

**Propósito.** Comprobar que sabe realizar la exploración básica de una colección de textos: tamaño y distribución, primer paso para decidir cómo procesarlos.

**Enunciado.** Considere que una palabra es cada elemento que resulta de separar el texto por espacios. Determine el número de documentos, el total de palabras, el índice (desde 0) del documento más largo en palabras (ante empate, el menor índice) y el promedio de palabras por documento (2 decimales).

**Formato de respuesta:** diccionario con `n_documentos` (int), `palabras_totales` (int), `indice_documento_mas_largo` (int) y `promedio_palabras` (float).

In [4]:
# Escriba su solución
RESP = {}
def registrar(n, valor):
    RESP[n] = valor
    print(json.dumps(valor, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))

n_documentos = len(documentos)
palabras_por_doc = [len(doc.split()) for doc in documentos]
palabras_totales = sum(palabras_por_doc)
indice_mas_largo = palabras_por_doc.index(max(palabras_por_doc))
promedio_palabras = round(palabras_totales / n_documentos, 2)

respuesta_1 = {
    "n_documentos": int(n_documentos),
    "palabras_totales": int(palabras_totales),
    "indice_documento_mas_largo": int(indice_mas_largo),
    "promedio_palabras": float(promedio_palabras)
}

registrar(1, respuesta_1)

{
 "n_documentos": 150,
 "palabras_totales": 3379,
 "indice_documento_mas_largo": 45,
 "promedio_palabras": 22.53
}


## Ejercicio 2. Normalización de texto (2 puntos)

**Propósito.** Evaluar el preprocesamiento de texto, etapa que decide la calidad de todo análisis posterior (frecuencias, similitud, clasificación).

**Enunciado.** Escriba la función `normalizar(texto)` que aplique, en este orden: (1) pasar a minúsculas, (2) eliminar URL que empiezan con `http://` o `https://`, (3) eliminar direcciones de correo con la forma `texto@texto.ec`, (4) reemplazar por espacio todo carácter que no sea letra (incluidas tildes, ñ y ü) ni espacio, (5) colapsar espacios múltiples y quitar los de los extremos.

Aplíquela al documento con índice 3.

**Formato de respuesta:** el texto normalizado (str).

In [5]:
# Escriba su solución
def normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r"https?://\S+", "", texto)
    texto = re.sub(r"\b\S+@\S+\.ec\b", "", texto)
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

respuesta_2 = normalizar(documentos[3])

registrar(2, respuesta_2)

"no recibí la factura del cobro mi pedido es ped mi correo es compré el detalle en"


## Ejercicio 3. Frecuencia de palabras (2 puntos)

**Propósito.** Evaluar la tokenización y el conteo de palabras, base de la representación de texto como números.

**Enunciado.** Defina la **tokenización** de un documento así: aplicar `normalizar` (ejercicio 2), separar por espacios, conservar solo las palabras de **3 o más letras** que **no estén** en `STOPWORDS`.

Obtenga las **5 palabras más frecuentes** en todo el corpus (ante empate, orden alfabético).

**Formato de respuesta:** lista de 5 pares `[palabra, frecuencia]`, de mayor a menor.

In [6]:
# Escriba su solución
def tokenizar(texto):
    norm = normalizar(texto)
    palabras = norm.split()
    return [p for p in palabras if len(p) >= 3 and p not in STOPWORDS]

todos_tokens = []
for doc in documentos:
    todos_tokens.extend(tokenizar(doc))

conteo = Counter(todos_tokens)
ordenados = sorted(conteo.items(), key=lambda x: (-x[1], x[0]))
top5 = [[str(w), int(c)] for w, c in ordenados[:5]]

respuesta_3 = top5

registrar(3, respuesta_3)

[
 [
  "llegó",
  88
 ],
 [
  "producto",
  40
 ],
 [
  "paquete",
  37
 ],
 [
  "perfecto",
  36
 ],
 [
  "soporte",
  36
 ]
]


## Ejercicio 4. Extracción de entidades con expresiones regulares (2 puntos)

**Propósito.** Evaluar la extracción de datos estructurados (correos, teléfonos, identificadores, fechas) desde texto libre, que es la forma más directa de convertir texto en columnas.

**Enunciado.** Sobre los textos **originales**, determine:

- cuántos **correos distintos** hay (formato `texto@correo.ec`, comparando en minúsculas),
- cuántos documentos contienen **al menos un teléfono** (`09` seguido de 8 dígitos, 10 dígitos en total),
- cuántos **identificadores de pedido distintos** hay (`PED-` y 5 dígitos),
- cuántas **fechas** en formato `dd/mm/aaaa` hay en total.

**Formato de respuesta:** diccionario con `correos_unicos`, `docs_con_telefono`, `pedidos_unicos` y `fechas_total` (todos int).

In [7]:
# Escriba su solución
correos = set()
for doc in documentos:
    found = re.findall(r"\b[a-zA-Z0-9._%+-]+@correo\.ec\b", doc, re.IGNORECASE)
    for c in found:
        correos.add(c.lower())

docs_con_telefono = sum(1 for doc in documentos if re.search(r"\b09\d{8}\b", doc))

pedidos = set()
for doc in documentos:
    found = re.findall(r"\bPED-\d{5}\b", doc, re.IGNORECASE)
    for p in found:
        pedidos.add(p.upper())

fechas_total = sum(len(re.findall(r"\b\d{2}/\d{2}/\d{4}\b", doc)) for doc in documentos)

respuesta_4 = {
    "correos_unicos": int(len(correos)),
    "docs_con_telefono": int(docs_con_telefono),
    "pedidos_unicos": int(len(pedidos)),
    "fechas_total": int(fechas_total)
}

registrar(4, respuesta_4)

{
 "correos_unicos": 37,
 "docs_con_telefono": 50,
 "pedidos_unicos": 121,
 "fechas_total": 42
}


## Ejercicio 5. Sentimiento con léxico (2 puntos)

**Propósito.** Evaluar la conversión de texto en una variable numérica (puntaje de sentimiento) usando listas de palabras, una técnica sencilla de análisis de opinión.

**Enunciado.** Para cada documento calcule `puntaje = (cantidad de tokens en POSITIVAS) - (cantidad de tokens en NEGATIVAS)`, usando la tokenización del ejercicio 3. Un documento es **positivo** si el puntaje es mayor que 0, **negativo** si es menor que 0 y **neutro** si es 0.

**Formato de respuesta:** diccionario con `positivos`, `negativos`, `neutros` (int), `indice_mayor_puntaje` (int, ante empate el menor índice) y `puntaje_maximo` (int).

In [8]:
# Escriba su solución
puntajes = []
for doc in documentos:
    tokens = tokenizar(doc)
    pos = sum(1 for t in tokens if t in POSITIVAS)
    neg = sum(1 for t in tokens if t in NEGATIVAS)
    puntajes.append(pos - neg)

positivos = sum(1 for p in puntajes if p > 0)
negativos = sum(1 for p in puntajes if p < 0)
neutros = sum(1 for p in puntajes if p == 0)

puntaje_maximo = max(puntajes)
indice_mayor_puntaje = puntajes.index(puntaje_maximo)

respuesta_5 = {
    "positivos": int(positivos),
    "negativos": int(negativos),
    "neutros": int(neutros),
    "indice_mayor_puntaje": int(indice_mayor_puntaje),
    "puntaje_maximo": int(puntaje_maximo)
}

registrar(5, respuesta_5)

{
 "positivos": 60,
 "negativos": 62,
 "neutros": 28,
 "indice_mayor_puntaje": 61,
 "puntaje_maximo": 6
}


## Ejercicio 6. Similitud entre documentos (2 puntos)

**Propósito.** Evaluar la representación de texto como bolsa de palabras y la medición de similitud con el coseno, base de los buscadores y los sistemas de recomendación.

**Enunciado.** Represente cada documento como un vector de conteos de tokens (tokenización del ejercicio 3) y use la **similitud coseno**: `(A . B) / (|A| x |B|)`.

Calcule (a) la similitud entre los documentos 1 y 2 y (b) el par `(i, j)` con `i < j` más similar entre los **primeros 12 documentos** (considere empate si la diferencia es menor que 1e-9; en ese caso, el menor `i` y luego el menor `j`).

**Formato de respuesta:** diccionario con `sim_1_2` (3 decimales), `par_mas_similar` (lista `[i, j]`) y `sim_par` (3 decimales).

In [9]:
# Escriba su solución
def similitud_coseno(c1, c2):
    inter = set(c1.keys()) & set(c2.keys())
    dot = sum(c1[w] * c2[w] for w in inter)
    norm1 = math.sqrt(sum(v**2 for v in c1.values()))
    norm2 = math.sqrt(sum(v**2 for v in c2.values()))
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

conteos = [Counter(tokenizar(doc)) for doc in documentos]

sim_1_2 = similitud_coseno(conteos[1], conteos[2])

max_sim = -1.0
mejor_par = None

for i in range(12):
    for j in range(i + 1, 12):
        s = similitud_coseno(conteos[i], conteos[j])
        if s > max_sim + 1e-9:
            max_sim = s
            mejor_par = [i, j]

respuesta_6 = {
    "sim_1_2": round(float(sim_1_2), 3),
    "par_mas_similar": mejor_par,
    "sim_par": round(float(max_sim), 3)
}

registrar(6, respuesta_6)

{
 "sim_1_2": 0.707,
 "par_mas_similar": [
  5,
  10
 ],
 "sim_par": 0.816
}


## Ejercicio 7. Palabra más distintiva (TF-IDF) (2 puntos)

**Propósito.** Evaluar la comprensión de TF-IDF, que destaca las palabras frecuentes en un documento y raras en el resto del corpus.

**Enunciado.** Para el documento con índice 10 calcule, para cada token, `tf = conteo en el documento / total de tokens del documento` e `idf = ln((1 + N) / (1 + df)) + 1`, donde `N` es el número total de documentos y `df` el número de documentos que contienen el token. Use `tf x idf` como puntaje.

Determine el token con mayor puntaje (ante empate, el primero en orden alfabético).

**Formato de respuesta:** diccionario con `termino` (str) y `tfidf` (4 decimales).

In [10]:
# Escriba su solución
docs_tokens = [tokenizar(doc) for doc in documentos]
N = len(docs_tokens)

tokens_10 = docs_tokens[10]
conteo_10 = Counter(tokens_10)
total_tokens_10 = len(tokens_10)

scores = {}
for term, count in conteo_10.items():
    tf = count / total_tokens_10
    df = sum(1 for dt in docs_tokens if term in dt)
    idf = math.log((1 + N) / (1 + df)) + 1
    scores[term] = tf * idf

ordenados = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
termino_top, tfidf_top = ordenados[0]

respuesta_7 = {
    "termino": str(termino_top),
    "tfidf": round(float(tfidf_top), 4)
}

registrar(7, respuesta_7)

{
 "termino": "descontó",
 "tfidf": 0.2477
}


## Ejercicio 8. Clasificación temática por reglas (2 puntos)

**Propósito.** Evaluar la creación de una variable categórica a partir de texto mediante reglas de palabras clave, equivalente a derivar una columna nueva.

**Enunciado.** Asigne a cada documento **una** categoría evaluando las reglas en este orden sobre el texto **en minúsculas** (una palabra clave coincide si aparece como subcadena):

1. `logistica`: contiene `envío`, `entrega`, `paquete`, `encomienda` o `retraso`.
2. `pagos`: contiene `pago`, `cobr`, `tarjeta`, `reembolso` o `factura`.
3. `producto`: contiene `producto`, `artículo` o `calidad`.
4. `atencion`: contiene `atención`, `asesor`, `soporte` o `chat`.
5. `otros`: ninguna de las anteriores.

**Formato de respuesta:** diccionario `{categoria: cantidad_de_documentos}` con las 5 categorías (incluya las que tengan 0).

In [11]:
# Escriba su solución
reglas = [
    ("logistica", ["envío", "entrega", "paquete", "encomienda", "retraso"]),
    ("pagos", ["pago", "cobr", "tarjeta", "reembolso", "factura"]),
    ("producto", ["producto", "artículo", "calidad"]),
    ("atencion", ["atención", "asesor", "soporte", "chat"])
]

conteo_cat = {"logistica": 0, "pagos": 0, "producto": 0, "atencion": 0, "otros": 0}

for doc in documentos:
    t = doc.lower()
    asignada = "otros"
    for cat, kws in reglas:
        if any(kw in t for kw in kws):
            asignada = cat
            break
    conteo_cat[asignada] += 1

respuesta_8 = conteo_cat

registrar(8, respuesta_8)

{
 "logistica": 60,
 "pagos": 47,
 "producto": 27,
 "atencion": 16,
 "otros": 0
}


## Ejercicio 9. La imagen como matriz (2 puntos)

**Propósito.** Evaluar la comprensión de que una imagen es una matriz de números y la extracción de características numéricas simples (brillo, color dominante, contraste).

**Enunciado.** Cada elemento de `imagenes` es una matriz `64 x 64 x 3` (canales rojo, verde y azul). Defina el gris como `0.299 R + 0.587 G + 0.114 B`.

Para cada imagen calcule el **brillo** (media del gris, 2 decimales), el **canal dominante** (`rojo`, `verde` o `azul`, el de mayor media) y el **contraste** (desviación estándar poblacional del gris). Indique además el índice de la imagen con mayor contraste.

**Formato de respuesta:** diccionario con `brillo` (lista de 5 floats), `canal_dominante` (lista de 5 str) y `mayor_contraste` (int).

In [12]:
# Escriba su solución
brillos = []
canales_dom = []
contrastes = []

for img in imagenes:
    R = img[:, :, 0].astype(float)
    G = img[:, :, 1].astype(float)
    B = img[:, :, 2].astype(float)
    
    gris = 0.299 * R + 0.587 * G + 0.114 * B
    
    brillos.append(round(float(np.mean(gris)), 2))
    
    means = {"rojo": np.mean(R), "verde": np.mean(G), "azul": np.mean(B)}
    canales_dom.append(str(max(means, key=means.get)))
    contrastes.append(float(np.std(gris)))

mayor_contraste = int(np.argmax(contrastes))

respuesta_9 = {
    "brillo": brillos,
    "canal_dominante": canales_dom,
    "mayor_contraste": mayor_contraste
}

registrar(9, respuesta_9)

{
 "brillo": [
  78.53,
  143.37,
  155.23,
  173.78,
  147.09
 ],
 "canal_dominante": [
  "rojo",
  "verde",
  "verde",
  "rojo",
  "verde"
 ],
 "mayor_contraste": 4
}


## Ejercicio 10. Huella visual y distancia entre imágenes (3 puntos)

**Propósito.** Evaluar la construcción de una representación compacta de una imagen (huella binaria) y su uso para medir parecido entre imágenes.

**Enunciado.** Construya la huella de cada imagen: (1) calcule su gris (ejercicio 9), (2) divídalo en **bloques de 8 x 8 píxeles** y calcule la media de cada bloque, obteniendo una matriz de 8 x 8, (3) cada bit es 1 si la media del bloque es **mayor** que la media de esa matriz de 8 x 8 y 0 en caso contrario (64 bits).

La **distancia de Hamming** entre dos huellas es el número de bits distintos. Calcule las distancias entre todas las imágenes y encuentre el par `(i, j)` con `i < j` de **menor distancia** (ante empate, el menor `i` y luego el menor `j`).

**Formato de respuesta:** diccionario con `par` (lista `[i, j]`), `distancia` (int) y `matriz` (lista de listas de 5 x 5 con las distancias).

In [13]:
# Escriba su solución
huellas = []

for img in imagenes:
    R = img[:, :, 0].astype(float)
    G = img[:, :, 1].astype(float)
    B = img[:, :, 2].astype(float)
    gris = 0.299 * R + 0.587 * G + 0.114 * B
    
    bloques = np.zeros((8, 8))
    for bi in range(8):
        for bj in range(8):
            bloques[bi, bj] = np.mean(gris[bi * 8:(bi + 1) * 8, bj * 8:(bj + 1) * 8])
            
    media_matriz = np.mean(bloques)
    bits = (bloques > media_matriz).astype(int).flatten()
    huellas.append(bits)

n_imgs = len(huellas)
matriz_dist = [[0] * n_imgs for _ in range(n_imgs)]
min_dist = 999
mejor_par = None

for i in range(n_imgs):
    for j in range(n_imgs):
        dist = int(np.sum(huellas[i] != huellas[j]))
        matriz_dist[i][j] = dist
        if i < j and dist < min_dist:
            min_dist = dist
            mejor_par = [i, j]

respuesta_10 = {
    "par": mejor_par,
    "distancia": int(min_dist),
    "matriz": matriz_dist
}

registrar(10, respuesta_10)

{
 "par": [
  1,
  2
 ],
 "distancia": 4,
 "matriz": [
  [
   0,
   57,
   55,
   53,
   21
  ],
  [
   57,
   0,
   4,
   18,
   46
  ],
  [
   55,
   4,
   0,
   20,
   42
  ],
  [
   53,
   18,
   20,
   0,
   36
  ],
  [
   21,
   46,
   42,
   36,
   0
  ]
 ]
}


## Ejercicio 11. Cruce entre categoría y sentimiento (2 puntos)

**Propósito.** Evaluar la integración de dos variables derivadas de texto (categoría y sentimiento) para responder una pregunta de negocio.

**Enunciado.** Con las categorías del ejercicio 8 y el sentimiento del ejercicio 5, calcule para cada categoría con **al menos 5 documentos** la proporción de documentos **negativos** (puntaje menor que 0), con 3 decimales. Indique la categoría con mayor proporción (ante empate, orden alfabético).

**Formato de respuesta:** diccionario con `categoria_mas_negativa` (str), `proporcion` (float) y `por_categoria` (diccionario categoría a proporción).

In [14]:
# Escriba su solución
categorias_doc = []
for doc in documentos:
    t = doc.lower()
    cat = "otros"
    for c, kws in reglas:
        if any(kw in t for kw in kws):
            cat = c
            break
    categorias_doc.append(cat)

totales_por_cat = Counter(categorias_doc)
negativos_por_cat = Counter()

for cat, p in zip(categorias_doc, puntajes):
    if p < 0:
        negativos_por_cat[cat] += 1

por_categoria = {}
for cat, tot in totales_por_cat.items():
    if tot >= 5:
        por_categoria[cat] = round(negativos_por_cat[cat] / tot, 3)

ordenadas = sorted(por_categoria.items(), key=lambda x: (-x[1], x[0]))
categoria_mas_negativa, max_prop = ordenadas[0]

respuesta_11 = {
    "categoria_mas_negativa": str(categoria_mas_negativa),
    "proporcion": float(max_prop),
    "por_categoria": por_categoria
}

registrar(11, respuesta_11)

{
 "categoria_mas_negativa": "logistica",
 "proporcion": 0.517,
 "por_categoria": {
  "producto": 0.259,
  "pagos": 0.383,
  "logistica": 0.517,
  "atencion": 0.375
 }
}


## Ejercicio 12. Deducción razonada (3 puntos)

**Propósito.** Evaluar la capacidad de interpretar los resultados de un análisis de datos no estructurados y de reconocer las limitaciones de los métodos usados.

**Enunciado.** Con base en los resultados de los ejercicios 3, 5, 8 y 11, redacte entre 5 y 8 líneas que expliquen **de qué tratan principalmente los mensajes y cómo es su tono**, y señale **al menos dos limitaciones** de los métodos usados (conteo de palabras, léxico de sentimiento y reglas de palabras clave). Incluya al menos **tres cifras** obtenidas.

**Formato de respuesta:** texto redactado en la celda de respuesta.

**Escriba aquí su respuesta:**

Los mensajes del corpus tratan fundamentalmente sobre el estado y entrega de pedidos (logistica lidera con 60 documentos y términos frecuentes como 'llegó' con 88 apariciones y 'paquete' con 37), seguidos de reclamos de facturación (pagos con 47) y consultas sobre artículos y servicio al cliente. El tono general de la clientela está marcadamente polarizado y tiende hacia la insatisfacción (62 documentos negativos frente a 60 positivos), siendo el área logística la más crítica al concentrar un 51.7% de opiniones desfavorables. No obstante, los métodos empleados presentan serias limitaciones: primero, el análisis basado en léxicos y bolsa de palabras ignora la estructura sintáctica y modificadores como la negación ('no fue rápido' se puntúa erróneamente positivo); segundo, las reglas de clasificación por subcadenas son rígidas y dependientes del orden jerárquico prefijado, lo que impide detectar ambigüedad temática o clasificar adecuadamente documentos con intenciones mixtas.